# Indexing MS MARCO v1 Document by OpenSearch for BM25 Model

- [msmarco-document](https://ir-datasets.com/msmarco-document.html)
- Prerequisite: corpus downloaded via [dataset/msmarco-v1-document](../../dataset/msmarco-v1-document/README.md)

BM25 indexes the **full** document body (no truncation) plus the title.

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv

In [ ]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

### Index a Corpus for BM25 Model

In [ ]:
import ir_datasets
dataset_name = "msmarco-document"
dataset = ir_datasets.load(dataset_name)

In [ ]:
index_name = "msmarco_v1_document_bm25"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      # Disabled during bulk indexing; re-enabled after the run below.
      "refresh_interval": "-1"
    }
    # English corpus: rely on OpenSearch's default (standard) analyzer.
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "url": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

Indexing (3.2M full documents, no server-side pipeline)

In [ ]:
def prepare_documents(dataset):
    """
    Yield raw bulk actions. MS MARCO documents are full web pages
    (doc_id, url, title, body).
    """
    for doc in dataset.docs_iter():
        text = doc.body.replace("\n", " ")
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "url": doc.url,
                "title": doc.title,
                "text": text,
            }
        }

In [ ]:
from opensearchpy.helpers import parallel_bulk

total = dataset.docs_count()   # 3,213,835

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in parallel_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        chunk_size=500,
        thread_count=4,
        queue_size=4,
        request_timeout=600,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed document
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)}")
if errors:
    pprint.pprint(errors[:3])          # inspect the first few errors

# Re-enable refresh now that bulk indexing is done, and make docs searchable.
client.indices.put_settings(index=index_name, body={"index": {"refresh_interval": "1s"}})
client.indices.refresh(index=index_name)
print("final count:", client.count(index=index_name)["count"])

---
### (Optional) Re-index documents missing from the first pass

If a run was interrupted, diff the corpus against what's actually in the index
and re-index just the missing ids.

At sustained multi-hour load, a small percentage of bulk items can fail with
transient `Error communicating with remote model: Connection reset`
(connection-level drops under parallel pressure; the model server itself
stays healthy). This is expected — the cells below recover exactly those
documents.

In [ ]:
from opensearchpy.helpers import scan

# All ids actually in the index (_source disabled -> fast).
indexed = set()
for hit in scan(
    client,
    index=index_name,
    query={"query": {"match_all": {}}, "_source": False},
    size=5000,
):
    indexed.add(hit["_id"])

# All ids the dataset should have produced
all_ids = {doc.doc_id for doc in dataset.docs_iter()}

missing = sorted(all_ids - indexed)
print(f"indexed: {len(indexed)},  missing: {len(missing)}")
print(missing[:10])

In [ ]:
# Re-index the documents identified as missing by the scan diff, printing every error.
from opensearchpy.helpers import parallel_bulk

def prepare_missing(dataset, missing_ids):
    docstore = dataset.docs_store()
    for doc_id in missing_ids:
        doc = docstore.get(doc_id)
        text = doc.body.replace("\n", " ")
        yield {
            "_id": doc_id,
            "_source": {"docid": doc_id, "url": doc.url, "title": doc.title, "text": text},
        }

retry_ok, retry_failed = 0, []
with tqdm(total=len(missing), desc="Re-indexing") as bar:
    for ok, item in parallel_bulk(
        client,
        prepare_missing(dataset, missing),
        index=index_name,
        chunk_size=128,
        thread_count=4,
        queue_size=4,
        request_timeout=600,
        raise_on_error=False,
        raise_on_exception=False,
    ):
        bar.update(1)
        retry_ok += ok
        if not ok:
            retry_failed.append(item)

print(f"retried ok: {retry_ok}, still failing: {len(retry_failed)}\n")

for item in retry_failed[:10]:     # full error detail for the first failures
    pprint.pprint(item)
    print("-" * 80)